# REGENIE

This performs the actual GWAS via REGENIE.

In [ ]:
%%capture
## Python Package Import
import sys
import os 
import numpy as np
import pandas as pd
from datetime import datetime

##Ensuring dsub is up to date
!pip3 install --upgrade dsub

#Defining environment variables
# Save this Python variable as an environment variable so that its easier to use within %%bash cells.
%env JOB_ID={LINE_COUNT_JOB_ID}
## Defining necessary pathways
my_bucket = os.environ['WORKSPACE_BUCKET']
project_name='HCM_GWAS_V2'
## Setting for running dsub jobs
pd.set_option('display.max_colwidth', 0)

USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.','-')

# Save this Python variable as an environment variable so that its easier to use within %%bash cells.
%env USER_NAME={USER_NAME}
%env PROJECT_NAME=HCM_GWAS_V2

In [ ]:
%%capture
JOB_NAME='REGENIE'

# Save this Python variable as an environment variable so that its easier to use within %%bash cells.
%env JOB_NAME={JOB_NAME}
%env PHENO_FILEPATH={PHENO_FILEPATH}
%env COV_FILEPATH={COV_FILEPATH}

## Analysis Results Folder 
line_count_results_folder = os.path.join(
    os.getenv('WORKSPACE_BUCKET'),
    'dsub',
    'results',
    os.getenv('PROJECT_NAME'),
    JOB_NAME,
    datetime.now().strftime('%Y%m%d'))

## Where the output files will go
output_files = os.path.join(line_count_results_folder)

OUTPUT_FILES = output_files

# Save this Python variable as an environment variable so that its easier to use within %%bash cells.
%env OUTPUT_FILES={OUTPUT_FILES}

## Step 1

This performs Step1 of REGENIE i.e whole-genome ridge regression.

In [ ]:
#This is the actual REGENIE bash script

filename2='aux_scripts/4_REGENIE_step1.sh'

script = '''
set -o pipefail 
set -o errexit

echo "${bedfile}"
echo "${bimfile}"
echo "${famfile}"

bfile_prefix="/mnt/data/input/gs/${MY_BUCKET}/dsub/results/HCM_GWAS_V2/REGENIE_PREP/20251107/array_regenie_step1_ldpruned_varprep"

regenie \
    --step 1 \
    --bed "${bfile_prefix}" \
    --keep "${keep_samples}" \
    --phenoFile "${pheno_file}" \
    --phenoCol "${pheno}" \
    --covarFile "${cov_file}" \
    --catCovarList "sex_at_birth,sample_source" \
    --covarColList "age_at_primary_consent,PC1,PC2,PC3,PC4,PC5,height,weight" \
    --bsize 1000 \
    --force-step1 \
    --verbose \
    --ref-first \
    --"${trait}" \
    --out "${pheno}"_step1

export regenie_step1_loco="${pheno}"_step1_1.loco
export regenie_step1_log="${pheno}"_step1.log
export regenie_step1_pred_list="${pheno}"_step1_pred.list

mv ${regenie_step1_pred_list} ${regenie_step1_log} ${regenie_step1_loco} -t ${OUTPUT_PATH}
'''

with open(filename2,'w') as fp:
    fp.write(script)
    
#Upload to GCP Bucket
!gsutil cp aux_scripts/4_REGENIE_step1.sh {my_bucket}/dsub/scripts/{project_name}/

In [ ]:
%%bash --out LINE_COUNT_JOB_ID

# Get a shorter username to leave more characters for the job name.
DSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1)"

AOU_NETWORK="global/networks/network"
AOU_SUBNETWORK="regions/us-central1/subnetworks/subnetwork"

MACHINE_TYPE="n2-standard-16" 
SCRIPT_NAME="4_REGENIE_step1.sh"
BASH_SCRIPT="${WORKSPACE_BUCKET}/dsub/scripts/HCM_GWAS_V2/${SCRIPT_NAME}"
BUCKET_BASENAME="${WORKSPACE_BUCKET#*gs://}" #Removes the gs:// prefix

dsub \
    --provider google-batch \
    --user-project "${GOOGLE_PROJECT}" \
    --project "${GOOGLE_PROJECT}" \
    --image "${ARTIFACT_REGISTRY_DOCKER_REPO}/jonchan0/regenie:v4.1.gz" \
    --network "${AOU_NETWORK}" \
    --subnetwork "${AOU_SUBNETWORK}" \
    --service-account "$(gcloud config get-value account)" \
    --user "${DSUB_USER_NAME}" \
    --regions us-central1 \
    --logging "${WORKSPACE_BUCKET}/dsub/logs/{job-name}/{user-id}/$(date +'%Y%m%d/%H%M%S')/{job-id}-{task-id}-{task-attempt}.log" \
    "$@" \
    --disk-size 1000 \
    --boot-disk-size 100 \
    --machine-type ${MACHINE_TYPE} \
    --name "REGENIE_step1" \
    --script "${BASH_SCRIPT}" \
    --env GOOGLE_PROJECT=${GOOGLE_PROJECT} \
    --input bedfile="${WORKSPACE_BUCKET}/dsub/results/HCM_GWAS_V2/REGENIE_PREP/20251107/array_regenie_step1_ldpruned_varprep.bed" \
    --input bimfile="${WORKSPACE_BUCKET}/dsub/results/HCM_GWAS_V2/REGENIE_PREP/20251107/array_regenie_step1_ldpruned_varprep.bim" \
    --input famfile="${WORKSPACE_BUCKET}/dsub/results/HCM_GWAS_V2/REGENIE_PREP/20251107/array_regenie_step1_ldpruned_varprep.fam" \
    --input keep_samples="${WORKSPACE_BUCKET}/HCM_GWAS_V2/3_output/3_REGENIE_PREP/3_regenie_files/hcm_gwas_ids.tsv" \
    --input pheno_file="${WORKSPACE_BUCKET}/HCM_GWAS_V2/3_output/3_REGENIE_PREP/3_regenie_files/hcm_gwas_regenie_pheno.tsv" \
    --input cov_file="${WORKSPACE_BUCKET}/HCM_GWAS_V2/3_output/3_REGENIE_PREP/3_regenie_files/hcm_gwas_regenie_cov.tsv" \
    --env pheno='hcm' \
    --env trait='bt' \
    --env MY_BUCKET=${BUCKET_BASENAME} \
    --use-private-address \
    --output-recursive OUTPUT_PATH="${OUTPUT_FILES}"

#N.B Make sure there are no spaces after the \ otherwise the dsub script breaks

In [ ]:
%%capture
!gsutil cp {my_bucket}/dsub/results/HCM_GWAS_V2/REGENIE/20251110/hcm_step1* ../3_output/4_REGENIE/1_step1/

In [ ]:
!gsutil -m cp -r ../3_output/4_REGENIE/1_step1/ {my_bucket}/HCM_GWAS_V2/3_output/4_REGENIE/

## Step 2

I run the GWAS i.e. step2 on both
1) Array data
2) srWGS ACAF data

### Array Dataset 

In [ ]:
filename2='aux_scripts/4_REGENIE_step2_array.sh'

script = '''
set -o pipefail 
set -o errexit

echo "${bedfile}"
echo "${bimfile}"
echo "${famfile}"

bfile_prefix=/mnt/data/input/gs/fc-aou-datasets-controlled/v8/microarray/plink/arrays

regenie \
    --step 2 \
    --bed "${bfile_prefix}" \
    --keep "${keep_samples}" \
    --extract "${extract_variants}" \
    --phenoFile "${pheno_file}" \
    --phenoCol "${pheno}" \
    --covarFile "${cov_file}" \
    --catCovarList "sex_at_birth,sample_source" \
    --covarColList "age_at_primary_consent,PC1,PC2,PC3,PC4,PC5,height,weight" \
    --pred "${step1_pred_list}" \
    --bsize 1000 \
    --minMAC 10 \
    --firth --approx \
    --verbose \
    --"${trait}" \
    --ref-first \
    --out "${pheno}"_step2_array

export regenie_log="${pheno}"_step2_array.log
export regenie_results="${pheno}"_step2_array_"${pheno}".regenie
mv ${regenie_results} ${regenie_log} -t ${OUTPUT_PATH}
'''

with open(filename2,'w') as fp:
    fp.write(script)
    
#Upload to GCP Bucket
!gsutil cp aux_scripts/4_REGENIE_step2_array.sh {my_bucket}/dsub/scripts/{project_name}/

In [ ]:
%%bash --out LINE_COUNT_JOB_ID

# Get a shorter username to leave more characters for the job name.
DSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1)"

AOU_NETWORK="global/networks/network"
AOU_SUBNETWORK="regions/us-central1/subnetworks/subnetwork"

MACHINE_TYPE="n2-standard-16" 
SCRIPT_NAME="4_REGENIE_step2_array.sh"
BASH_SCRIPT="${WORKSPACE_BUCKET}/dsub/scripts/HCM_GWAS_V2/${SCRIPT_NAME}"

dsub \
    --provider google-batch \
    --user-project "${GOOGLE_PROJECT}" \
    --project "${GOOGLE_PROJECT}" \
    --image "${ARTIFACT_REGISTRY_DOCKER_REPO}/jonchan0/regenie:v4.1.gz" \
    --network "${AOU_NETWORK}" \
    --subnetwork "${AOU_SUBNETWORK}" \
    --service-account "$(gcloud config get-value account)" \
    --user "${DSUB_USER_NAME}" \
    --regions us-central1 \
    --logging "${WORKSPACE_BUCKET}/dsub/logs/{job-name}/{user-id}/$(date +'%Y%m%d/%H%M%S')/{job-id}-{task-id}-{task-attempt}.log" \
    "$@" \
    --disk-size 1000 \
    --boot-disk-size 100 \
    --machine-type ${MACHINE_TYPE} \
    --name "REGENIE_step2_array" \
    --script "${BASH_SCRIPT}" \
    --env GOOGLE_PROJECT=${GOOGLE_PROJECT} \
    --input bedfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bed" \
    --input bimfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bim" \
    --input famfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.fam" \
    --input keep_samples="${WORKSPACE_BUCKET}/HCM_GWAS_V2/3_output/3_REGENIE_PREP/3_regenie_files/hcm_gwas_ids.tsv" \
    --input extract_variants="${WORKSPACE_BUCKET}/HCM_GWAS_V2/3_output/2_VARIANT_QC/array/array_variantqc_pass_final_vars.txt" \
    --input pheno_file="${WORKSPACE_BUCKET}/HCM_GWAS_V2/3_output/3_REGENIE_PREP/3_regenie_files/hcm_gwas_regenie_pheno.tsv" \
    --input cov_file="${WORKSPACE_BUCKET}/HCM_GWAS_V2/3_output/3_REGENIE_PREP/3_regenie_files/hcm_gwas_regenie_cov.tsv" \
    --input step1_loco="${WORKSPACE_BUCKET}/HCM_GWAS_V2/3_output/4_REGENIE/1_step1/hcm_step1_1.loco" \
    --input step1_pred_list="${WORKSPACE_BUCKET}/HCM_GWAS_V2/3_output/4_REGENIE/1_step1/hcm_step1_pred_adj.list" \
    --env pheno='hcm' \
    --env trait='bt' \
    --use-private-address \
    --output-recursive OUTPUT_PATH="${OUTPUT_FILES}"

#N.B Make sure there are no spaces after the \ otherwise the dsub script breaks

### srWGS ACAF Dataset

In [ ]:
filename2='aux_scripts/4_REGENIE_step2_srWGS_ACAF.sh'

script = '''
set -o pipefail 
set -o errexit
#This defines the actual bed_prefix, assuming localisation of the input bed/bim/fam files

echo "${bedfile}"
echo "${bimfile}"
echo "${famfile}"

bed_prefix=/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr"${chrom}" 


regenie \
    --step 2 \
    --bed "${bed_prefix}" \
    --keep "${keep_samples}" \
    --extract "${extract_variants}" \
    --phenoFile "${pheno_file}" \
    --phenoCol "${pheno}" \
    --covarFile "${cov_file}" \
    --catCovarList "sex_at_birth,sample_source" \
    --covarColList "age_at_primary_consent,PC1,PC2,PC3,PC4,PC5,height,weight" \
    --pred "${step1_pred_list}" \
    --bsize 1000 \
    --minMAC 10 \
    --firth --approx \
    --verbose \
    --"${trait}" \
    --ref-first \
    --out "${pheno}"_step2_chr"${chrom}"

export regenie_log="${pheno}"_step2_chr"${chrom}".log
export regenie_results="${pheno}"_step2_chr"${chrom}"_"${pheno}".regenie
mv ${regenie_results} ${regenie_log} -t ${OUTPUT_PATH}
'''

with open(filename2,'w') as fp:
    fp.write(script)
    
#Upload to GCP Bucket
!gsutil cp aux_scripts/4_REGENIE_step2_srWGS_ACAF.sh {my_bucket}/dsub/scripts/{project_name}/

In [ ]:
%%bash --out LINE_COUNT_JOB_ID

# Get a shorter username to leave more characters for the job name.
DSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1)"

AOU_NETWORK="global/networks/network"
AOU_SUBNETWORK="regions/us-central1/subnetworks/subnetwork"

MACHINE_TYPE="n2-standard-16" 
SCRIPT_NAME="4_REGENIE_step2_srWGS_ACAF.sh"
BASH_SCRIPT="${WORKSPACE_BUCKET}/dsub/scripts/HCM_GWAS_V2/${SCRIPT_NAME}"

#Test on 1 chromosome first
LOWER=1
UPPER=22
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
    dsub \
        --provider google-batch \
        --user-project "${GOOGLE_PROJECT}" \
        --project "${GOOGLE_PROJECT}" \
    --image "${ARTIFACT_REGISTRY_DOCKER_REPO}/jonchan0/regenie:v4.1.gz" \
        --network "${AOU_NETWORK}" \
        --subnetwork "${AOU_SUBNETWORK}" \
        --service-account "$(gcloud config get-value account)" \
        --user "${DSUB_USER_NAME}" \
        --regions us-central1 \
        --logging "${WORKSPACE_BUCKET}/dsub/logs/{job-name}/{user-id}/$(date +'%Y%m%d/%H%M%S')/{job-id}-{task-id}-{task-attempt}.log" \
        "$@" \
        --disk-size 1000 \
        --boot-disk-size 100 \
        --machine-type ${MACHINE_TYPE} \
        --name "REGENIE_step2_srWGS_ACAF_chr${chromo}" \
        --script "${BASH_SCRIPT}" \
        --env GOOGLE_PROJECT=${GOOGLE_PROJECT} \
        --input bedfile="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bed" \
        --input bimfile="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bim" \
        --input famfile="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.fam" \
        --input keep_samples="${WORKSPACE_BUCKET}/HCM_GWAS_V2/3_output/3_REGENIE_PREP/3_regenie_files/hcm_gwas_ids.tsv" \
        --input extract_variants="${WORKSPACE_BUCKET}/HCM_GWAS_V2/3_output/2_VARIANT_QC/srWGS_ACAF/qc1_hwepass_vars/srWGS_ACAF_qc1_hwepass_vars_chr${chromo}.txt" \
        --input pheno_file="${WORKSPACE_BUCKET}/HCM_GWAS_V2/3_output/3_REGENIE_PREP/3_regenie_files/hcm_gwas_regenie_pheno.tsv" \
        --input cov_file="${WORKSPACE_BUCKET}/HCM_GWAS_V2/3_output/3_REGENIE_PREP/3_regenie_files/hcm_gwas_regenie_cov.tsv" \
        --input step1_loco="${WORKSPACE_BUCKET}/HCM_GWAS_V2/3_output/4_REGENIE/1_step1/hcm_step1_1.loco" \
        --input step1_pred_list="${WORKSPACE_BUCKET}/HCM_GWAS_V2/3_output/4_REGENIE/1_step1/hcm_step1_pred_adj.list" \
        --env pheno='hcm' \
        --env trait='bt' \
        --env chrom=${chromo} \
        --use-private-address \
        --output-recursive OUTPUT_PATH="${OUTPUT_FILES}"
done

#N.B Make sure there are no spaces after the \ otherwise the dsub script breaks

In [ ]:
%%capture
!gsutil -m cp {my_bucket}/dsub/results/HCM_GWAS_V2/REGENIE/20251110/hcm_step2_chr* ../3_output/4_REGENIE/2_step2/srWGS_ACAF/

## Dstat